# Field age: instrument feasibility

## Question and why it matters
Does field maturation explain return hazards?

This notebook reports instrument calibration and a prospective feasibility pilot. Scientific confirmation is not run. See [README](README.md) and [mechanism note](../../../notes/kappa-mechanisms-2026-10-15.md).

## Assumptions and definitions
Discovery includes first arrival; visit count means departures, including from the initial patch. Phase is (initial colour + departures) mod rule length. Completed cycles are floor(departures/n), without inventing prehistory for initial patch colours. The visited boundary includes hole edges; rim depth 0..4 is graph distance to that boundary. Census features weight cells; hazard tables weight departures. Exact fixture/property checks support these definitions. Return-hazard estimates remain observational.

## Provenance
All inputs below are small results intended for version control; this session makes no commit. The notebook never reads dumps or runs simulation. Seed identities are printed from the frozen manifest.

In [1]:
from pathlib import Path
import hashlib, json, csv
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Cargo.toml").is_file() and (p / "experiments").is_dir())
EXP = ROOT / "experiments/archive/2026-10-kappa-field-age"
inputs = ["pilot-v2.manifest.json", "pilot_summary.json", "pilot_features.csv", "calibration.json", "determinism.json", "REPORT.md"] + (["cohort_audit.json"] if EXP.name.endswith("field-age") else [])
for name in inputs:
    path = EXP / "results" / name
    assert path.is_file(), path
    print(name, hashlib.sha256(path.read_bytes()).hexdigest())
manifest = json.loads((EXP / "results/pilot-v2.manifest.json").read_text())
print("Simulation base commit:", manifest["git_commit"])
print("Registered UTC:", manifest["recorded_utc"])
print("Pilot seeds:", manifest["config"]["seeds"])
SEED = manifest["config"]["seeds"][0]


pilot-v2.manifest.json 2ce03575f8a71a6d4ef7eb58be48d16e0370ff1c28ecfa71855363190ede298b
pilot_summary.json efc3a528c12c43f5c52ec44aa6b7e7a8c11af24e947e5a73da7de0d01951a2e9
pilot_features.csv cd747c2931ed6416534ae8373c7dd936930d3bf42631c7f088d192812f6f9167
calibration.json fa83323dae78dcb5de6bb6e4c6b5c0b6343f23c2dcb1f43e039c5df9d304b6df
determinism.json f2c09343e67cb98c892269900ee1d19c23af88b557bdddb770a0e0fd10528714
REPORT.md 265d98d6a627b68cd9a8fa9ba5d4a25432a38e6f96706df59302cd51153d8ca6
cohort_audit.json 9594dc31a2e99350e89b3caefd0b5b02830f46e93ac921a35c9d43afb7d9ed99
Simulation base commit: b3a0f762ac2230b5df92188206a4d3a1081897be
Registered UTC: 2026-10-15T10:12:25.860688+00:00
Pilot seeds: [2026101511, 2026101512, 2026101513, 2026101514]


## Data and method
The Rust observation driver preserves the original walker and feeds the tested fixed-area accountant. Interior episodes intersecting a band are followed beyond its end; full durations and in-band overlap have separate columns. An unresolved return has no assumed finite upper bound. Birth, midpoint and endpoint censuses enumerate the visited rim. Heavy computation lives in the example binary and scripts/pilot.py. Matching and all seeds were fixed before new ant outcomes.

## Results
Frozen pilot: 44 rules x 4 starts, three area bands; cap 2^30, grid 4096, four workers. Base commit `b3a0f762ac2230b5df92188206a4d3a1081897be`; dirty source hashes and seed identities are in [manifest](results/pilot-v2.manifest.json).

| Band start | Completed / requested | Unresolved interior time | Minimum completed entry episodes |
|---|---:|---:|---:|
| 32768 | 176 / 176 | 0 | 1018 |
| 131072 | 176 / 176 | 0 | 4085 |
| 524288 | 168 / 176 | 0 | 0 |

No-escape gate: **True**. All-complete gate: **False**. Quarter-cap reserve gate: **False** (max final step 1,073,741,824). Ant batch wall time 757.9 s; measured whole-process peak RSS 514.2 MiB (four concurrent jobs).

## Instrument validation

Four reference/property tests check discovery times, exact departure counts, nonzero patch phase offsets, visited-boundary rim census, offline fragment accounting, follow-up past band edges, right censoring, and rotor/random-walk paths. Full release tests passed; see the tail-budget results/instrument-release-tests.log.

Sampled thread replay: **True**, 8 specs and 48 files, one versus six workers at the two profile bands. Finished-trajectory resume preserves bytes. Full all-seed replay and mid-trajectory state checkpoints are **not** validated.

Existing pool support before matching: 169 stationary and 23 drifting rules. The latter is below the required 24 even without matching; exact length/prefix matching retains 22 pairs. See cohort_audit.json.

## Coverage by pre-existing stratum

| Stratum | Area 32768 | Area 131072 | Area 524288 |
|---|---:|---:|---:|
| stationary | 88 / 88 | 88 / 88 | 88 / 88 |
| drifting | 88 / 88 | 88 / 88 | 80 / 88 |

Synthetic exogenous-age chain (N=400,000, seed 2026101581): known return-hazard contrast 0.24, estimated 0.239944; after within-stratum label permutation 0.002478. Both four-standard-error calibration checks pass. This calibrates transition-frequency estimation, not a kappa model or a physical ageing intervention.

**Original cohort feasibility gate failed:** FAILED: only 22 exact matched pairs in existing pool. The pilot uses the 44 supported rules (22 exact length/prefix-matched pairs). This does not silently replace the planned 48-rule confirmation cohort.

## Noncomplete trajectories

These remain in every coverage denominator; missing-band features are null, not zero effects.

- `RLLLLLLRRRRR@2026101511`: budget, final step 1073741824, visited cells 408316.
- `RLLLLLLRRRRR@2026101512`: budget, final step 1073741824, visited cells 397468.
- `RLLLLLLRRRRR@2026101513`: budget, final step 1073741824, visited cells 392092.
- `RLLLLLLRRRRR@2026101514`: budget, final step 1073741824, visited cells 403732.
- `RLLLLRRLLRRR@2026101511`: budget, final step 1073741824, visited cells 498388.
- `RLLLLRRLLRRR@2026101512`: budget, final step 1073741824, visited cells 503496.
- `RLLLLRRLLRRR@2026101513`: budget, final step 1073741824, visited cells 492960.
- `RLLLLRRLLRRR@2026101514`: budget, final step 1073741824, visited cells 500324.

## Split-seed feature reliability (descriptive)

Spearman correlation across rules between first two and last two pilot seeds, restricted to rules with all four starts attaining the band and the feature observed. Contributing rule counts are shown in parentheses. No bootstrap inference is made; coverage denominators retain every requested start. The CSV includes every seed and between-seed dispersion is in the summary JSON.

| Feature | Area 32768 | Area 131072 | Area 524288 |
|---|---:|---:|---:|
| immature | 0.916 (44) | 0.847 (44) | 0.860 (42) |
| median_log_age | 0.413 (44) | 0.321 (44) | 0.718 (42) |
| cycle_variance | 0.910 (44) | 0.869 (44) | 0.846 (42) |

## Meaning and remaining work

These results establish observation/accounting feasibility at the measured scales and describe repeatability. They do not test T1/T2 or A1/A2/A3, compare tail or age features with kappa, establish causal mechanisms, or calibrate a survival-family fit or drift forecast.

Confirmation remains unregistered and unlaunched. Before confirmation: implement and byte-test mid-trajectory state checkpoints; calibrate/freeze the survival family and predictor loss; freeze grouped rule folds and seed roles; resolve the failed age cohort support gate explicitly; choose seed counts using pilot feature reliability. Retain the original thresholds and report deviations prospectively. No successful scientific prediction is claimed.

## Reproduce

Use WSL from the repo root with `export UV_PROJECT_ENVIRONMENT=$HOME/.venvs/turmite UV_LINK_MODE=copy`. Source-hash guards reject changes to registered inputs; retain old manifests and use a new run label for changed sources.

```bash
cargo build --release --example kappa_tail_budget_sampler --example kappa_field_age_sampler --example kappa_tail_budget_calibrate
uv run experiments/archive/2026-10-kappa-field-age/scripts/calibrate.py
uv run experiments/archive/2026-10-kappa-field-age/scripts/run.py --name pilot-v2
uv run experiments/archive/2026-10-kappa-field-age/scripts/summarize.py
uv run experiments/archive/2026-10-kappa-field-age/scripts/verify.py
uv run experiments/archive/2026-10-kappa-field-age/scripts/report.py
cargo test --release
```

The pilot itself is frozen prospectively in pilot-v2.manifest.json. Its exact preparation command is recorded there. pilot-v1 was rejected before simulation because an unrelated module export changed src/lib.rs; its manifest and failure log are retained.


In [2]:
summary = json.loads((EXP / "results/pilot_summary.json").read_text())
print("Run gates:", json.dumps(summary["run_gates"], indent=2))
for b in summary["bands"]:
    print("Area", b["lo"], "completed", b["completed"], "/", b["requested"],
          "unresolved in-band I steps", b["unresolved_overlap"])
print("Scientific verdict:", summary["scientific_verdict"])


Run gates: {
  "all_complete": false,
  "elapsed_seconds": 757.8620079959983,
  "max_t": 1073741824,
  "no_escape": true,
  "quarter_cap_reserve": false
}
Area 32768 completed 176 / 176 unresolved in-band I steps 0
Area 131072 completed 176 / 176 unresolved in-band I steps 0
Area 524288 completed 168 / 176 unresolved in-band I steps 0
Scientific verdict: NOT TESTED: no fitted/calibrated survival family, outcome seed roles, or frozen confirmatory regression; no causal conclusion


The tables show instrument coverage and repeatability at these scales. They do not show that either mechanism predicts independent-seed kappa.

## Caveats and controls
Rotor and random walk are path/accounting controls at smaller areas, not a universal kappa calibration. Synthetic hazard labels are exogenous; label permutation is not a physical field intervention. Full state checkpoints and a frozen scientific estimator remain missing. The original age cohort matching gate failed.

## What this changes / next steps
Observation-only pilots are now feasible with measured resource cost. Freeze the estimator and resolve the cohort and checkpoint requirements before confirmation.

## Open questions
Does tail probability or conditional tail mean transfer independently? Do cell-weighted age features add predictive value beyond elapsed time, geometry, and prefix on held-out rules?